# POSE GENERATION

In [ ]:
# IMPORTS

%load_ext autoreload
%autoreload 2
import hippo
import hippo.pset as pset
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import plotly.express as px
pset.mp = mp
mrich.patch_rich_jupyter_margins()

## Renitialise Directories for Current Cycle

In [ ]:
target_name = "Target Name"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Increment this for each new design cycle for this target.
method_tags = ["fragmenstein", "pure_knitwork", "impure_knitwork"] # Select the elaboration methods you want to analyse. These should match the method tags used in previous steps.
score_col = "energy_score"  # column in score_df to rank poses by; also available: "distance_score", "cnn_vs"
score_quantile = 0.2  # keep the best-scoring 20% of poses for each method
selection_tag = f"c{cycle_number:02}_selected"  # tag applied to the shortlisted poses

# Change these to your own details for Fragalysis submission.

YOUR_NAME = "Your Name"
YOUR_EMAIL = "your.email@example.com"
YOUR_INSTITUTION = "Your Institution"

In [ ]:
# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
fragmenstein_dir = cycle_dir / "fragmenstein"
knitwork_dir = cycle_dir / "knitwork"
knitwork_pure_output_dir = knitwork_dir / "knitwork_pure_output"
knitwork_impure_output_dir = knitwork_dir / "knitwork_impure_output"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_inputs_fragmenstein_dir =  gnina_inputs_dir / "fragmenstein"
gnina_inputs_knitwork_pure_dir =  gnina_inputs_dir / "knitwork_pure"
gnina_inputs_knitwork_impure_dir =  gnina_inputs_dir / "knitwork_impure"
gnina_outputs_dir = gnina_dir / "outputs"
gnina_outputs_fragmenstein_dir =  gnina_outputs_dir / "fragmenstein"
gnina_outputs_knitwork_pure_dir =  gnina_outputs_dir / "knitwork_pure"
gnina_outputs_knitwork_impure_dir =  gnina_outputs_dir / "knitwork_impure"
moccassin_outputs_dir = cycle_dir / "moccassin_outputs"
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
bulk_target_dir = bulk_targets_dir / target_name

## Generate environment variable to run command directly from the Notebook

In [ ]:
environ["TARGET_NAME"] = target_name
environ["FRAGMENSTEIN_DIR"] = str(fragmenstein_dir.resolve())
environ["KNITWORK_DIR"] = str(knitwork_dir.resolve())

# Run BulkDock Placement

## Generate BulkDock inputs

### Fragmenstein

Make sure the cell bellow outputs the full path of the `fragmenstein` directory:

In [ ]:
!echo $FRAGMENSTEIN_DIR

In [ ]:
python $XCHEM_FFF/scripts/fragmenstein_to_bulkdock.py $FRAGMENSTEIN_DIR

In [ ]:
!cp ${FRAGMENSTEIN_DIR}/fragmenstein_bulkdock_input.csv ${BULK}/INPUTS/${target_name}_fragmenstein.csv

### Knitwork

In [ ]:
!echo $KNITWORK_DIR

Make sure the cell bellow outputs the full path of the `knitwork` directory:

In [ ]:
python $XCHEM_FFF/scripts/knitwork_to_bulkdock.py $KNITWORK_DIR

In [ ]:
!cp ${KNITWORK_DIR}/knitwork_pure_output/knitwork_pure_bulkdock_input.csv ${BULK}/INPUTS/${TARGET_NAME}_knitwork_pure.csv
!cp ${KNITWORK_DIR}/knitwork_impure_output/knitwork_impure_bulkdock_input.csv ${BULK}/INPUTS/${TARGET_NAME}_knitwork_impure.csv

## Run Placement Jobs 

Can be done here or directly from the terminal by removing the !

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_fragmenstein.csv --split 2000

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_pure_knitwork.csv --split 2000

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_impure_knitwork.csv --split 2000

Verify if your jobs are running with the `squeue` command:

In [ ]:
!squeue -u $USER

## Export Placed Poses for GNINA Minimisation

In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags #Display available tags

In [ ]:
fragmenstein_poses = animal.poses.get_by_tag("<target_name>_fragmenstein") #replace this with the relevant tag associated with your fragmenstein poses.

fragmenstein_poses.to_fragalysis(
    str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein.sdf"),
    method = "fragmenstein",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

In [ ]:
knitwork_pure_poses = animal.poses.get_by_tag("<target_name>_pure_knitwork") #replace this with the relevant tag associated with your pure knitwork poses.

knitwork_pure_poses.to_fragalysis(
    str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure.sdf"),
    method = "knitwork",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

In [ ]:
knitwork_impure_poses = animal.poses.get_by_tag("<target_name>_impure_knitwork") #replace this with the relevant tag associated with your impure knitwork poses.

knitwork_impure_poses.to_fragalysis(
    str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure.sdf"),
    method = "knitwork",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

## Run GNINA Minimisation and Scoring

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein.sdf")} \
--ref_pdbs {str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein_refs.zip")} \
--output_path {gnina_outputs_fragmenstein_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure.sdf")} \
--ref_pdbs {str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure_refs.zip")} \
--output_path {gnina_outputs_knitwork_pure_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure.sdf")} \
--ref_pdbs {str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure_refs.zip")} \
--output_path {gnina_outputs_knitwork_impure_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

Verify if your jobs are running with the `squeue` command:

In [ ]:
!squeue -u $USER

## Import GNINA minimised poses to HIPPO for visualisation.

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_fragmenstein_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "fragmenstein"],
        name_col=None
    )

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_knitwork_pure_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "pure_knitwork"],
        name_col=None
    )

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_knitwork_impure_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "impure_knitwork"],
        name_col=None
    )

## Export subset of PoseSet to Fragalysis

### 1. Build a score table for the GNINA-minimised poses

In [ ]:
gnina_poses = animal.poses.get_by_tag("gnina_repose")

def get_method(pose):
    # the method tag is only recorded on the pre-GNINA placed pose, not the GNINA-reposed
    # one, but load_sdf preserves the placed pose's tags in metadata["tags"]
    pose_tags = (pose.metadata.get("tags") or "").split(",")
    for method in method_tags:
        if f"{target_name}_{method}" in pose_tags:
            return method
    return None

data = []
for pose in gnina_poses:
    data.append({
        "pose_id": pose.id,
        "compound_id": pose.compound_id,
        "method": get_method(pose),
        "energy_score": pose.energy_score,
        "distance_score": pose.distance_score,
        # CNN_VS isn't a dedicated HIPPO score field, but GNINA's rescoring SDF columns
        # are preserved on each pose's metadata, so it can still be pulled out for triage.
        "cnn_vs": pose.metadata.get("CNN_VS"),
    })

score_df = pd.DataFrame(data).set_index("pose_id")
score_df.head()

In [ ]:
score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_scores.csv")

### 2. Visualise the score distributions

In [ ]:
px.histogram(score_df, x="energy_score", color="method", barmode="overlay")

In [ ]:
px.scatter(score_df, x="distance_score", y="energy_score", color="method", hover_data=["compound_id"])

In [ ]:
px.histogram(score_df, x="cnn_vs", color="method", barmode="overlay")

### 3. Select the top-scoring poses

In [ ]:
higher_is_better = {"cnn_vs"}  # cnn_vs favours larger values; energy_score/distance_score favour smaller ones

if score_col in higher_is_better:
    cutoffs = score_df.groupby("method")[score_col].quantile(1 - score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] >= cutoffs[row["method"]], axis=1)
    ]
else:
    cutoffs = score_df.groupby("method")[score_col].quantile(score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] <= cutoffs[row["method"]], axis=1)
    ]

top_score_df

In [ ]:
top_poses = animal.poses[list(top_score_df.index)]
top_poses

### 4. Generate pose overlays and images of the top and bottom poses

In [ ]:
for pose in top_poses[:5]:
    print(pose)
    pose.draw()  # 2D depiction of the pose's compound
for pose in top_poses[-5:]:
    print(pose)
    pose.draw()  # 2D depiction of the pose's compound

In [ ]:
# eyeball each top-scoring pose alongside its inspirations to sanity-check
# the placement/merge before committing to the selection
for pose in top_poses[:5]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()
for pose in top_poses[-5:]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()

### 5. Tag the top-scoring poses

After quick sanity checks, attribute the `selection_tag` (defined in the config section above) to the shortlisted poses.
Use the tag later to extract these poses from HIPPO to run elaborations, to export an SDF for Fragalysis review, or to pull out SMILES for compound ordering / recipe generation in CAR — via `animal.poses.get_by_tag(<selection_tag>)`

In [ ]:
top_poses.add_tag(selection_tag)
selection_tag

### 6. Export the tagged poses

In [ ]:
top_score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.csv")

In [ ]:
top_poses.to_fragalysis(
    str(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.sdf"),
    method = "gnina",
    submitter_name = YOUR_NAME,
    submitter_email = YOUR_EMAIL,
    submitter_institution = YOUR_INSTITUTION,
    copy_reference_pdbs=True
)

In [ ]:
# Export the SMILES of the shortlisted compounds, e.g. for compound ordering or recipe generation in CAR.
# CompoundSet.smiles/.ids return plain lists, so build a DataFrame before writing the CSV.

pd.DataFrame({
    "compound_id": top_poses.compounds.ids,
    "smiles": top_poses.compounds.smiles,
}).to_csv(
    gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_compounds_smiles.csv",
    index=False,
)